In [1]:
# Install libraries 
 
import subprocess, sys
pkgs = ["librosa", "soundfile", "tqdm", "pandas", "numpy",
        "scikit-learn", "matplotlib", "seaborn",
        "tensorflow", "opencv-python", "onnxruntime"]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
print("All packages installed ✓")

All packages installed ✓


In [3]:
# Clone YOLOv6 repo

import os, sys, subprocess

BASE      = r"C:\Users\Acer Swift X\Downloads\iot project"
YOLO_REPO = os.path.join(BASE, "YOLOv6")

# Clone repo if not already there
if not os.path.exists(YOLO_REPO):
    subprocess.run(["git", "clone",
                    "https://github.com/meituan/YOLOv6", YOLO_REPO], check=True)
    print("YOLOv6 cloned ✓")
else:
    print("YOLOv6 already cloned, skipping.")

# Install ONLY the packages needed for export — skip torch/tensorflow version pins
# that conflict with tfgpu environment
needed = [
    "onnx>=1.12",
    "onnxsim",          
    "onnxruntime",
    "PyYAML",
    "tqdm",
    "scipy",
]

for pkg in needed:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✓ {pkg}")
    else:
        print(f"  ✗ {pkg} failed: {result.stderr.strip()[-200:]}")

print("\nDone  (intentionally skipped full requirements.txt to avoid conflicts)")

YOLOv6 already cloned, skipping.
  ✓ onnx>=1.12
  ✗ onnxsim failed: nnxsim_bf6e315451a146608388bc7804ef0d4e\\third_party\\onnx-optimizer\\third_party\\onnx\\onnx\\backend\\test\\data\\node\\test_argmax_negative_axis_keepdims_example_select_last_index\\test_data_set_0'
  ✓ onnxruntime
  ✓ PyYAML
  ✓ tqdm
  ✓ scipy

Done ✓  (intentionally skipped full requirements.txt to avoid conflicts)


In [8]:
# Quick check
import os
deploy = r"C:\Users\Acer Swift X\Downloads\iot project\YOLOv6\deploy\ONNX"
print(os.listdir(deploy))

['eval_trt.py', 'export_onnx.py', 'OpenCV', 'README.md', 'YOLOv6-Dynamic-Batch-onnxruntime.ipynb', 'YOLOv6-Dynamic-Batch-tensorrt.ipynb']


In [ ]:
# Download YOLOv6 weights + export to ONNX

import urllib.request, shutil
 
BASE         = r"C:\Users\Acer Swift X\Downloads\iot project"
MODELS_DIR   = os.path.join(BASE, "models")
YOLO_PT      = os.path.join(MODELS_DIR, "yolov6n.pt")
YOLO_ONNX    = os.path.join(MODELS_DIR, "yolov6n.onnx")
YOLO_REPO    = os.path.join(BASE, "YOLOv6")
WEIGHTS_URL  = "https://github.com/meituan/YOLOv6/releases/download/0.4.0/yolov6n.pt"
 
os.makedirs(MODELS_DIR, exist_ok=True)
 
# Download .pt weights
if not os.path.exists(YOLO_PT):
    print("Downloading yolov6n.pt ...")
    urllib.request.urlretrieve(WEIGHTS_URL, YOLO_PT)
    print(f"Saved → {YOLO_PT} ✓")
else:
    print("yolov6n.pt already exists, skipping download.")
 
# Export .pt → .onnx
if not os.path.exists(YOLO_ONNX):
    print("Exporting to ONNX ...")
    export_script = os.path.join(YOLO_REPO, "deploy", "ONNX", "export_onnx.py") 
    
    # Add YOLOv6 repo to PYTHONPATH so yolov6 module can be found
    env = os.environ.copy()
    env["PYTHONPATH"] = YOLO_REPO + os.pathsep + env.get("PYTHONPATH", "")
    
    cmd = [sys.executable, export_script,
           "--weights", YOLO_PT,
           "--img", "640",
           "--device", "cpu",
           "--simplify"]
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    print(result.stdout)
    if result.returncode != 0:
        print("EXPORT ERROR:", result.stderr)
    else:
        # YOLOv6 saves the ONNX next to the .pt by default; move it to models/
        default_out = YOLO_PT.replace(".pt", ".onnx")
        if os.path.exists(default_out) and default_out != YOLO_ONNX:
            shutil.move(default_out, YOLO_ONNX)
        print(f"ONNX saved → {YOLO_ONNX} ✓")
else:
    print("yolov6n.onnx already exists, skipping export.")

yolov6n.pt already exists, skipping download.
Exporting to ONNX ...

EXPORT ERROR: Traceback (most recent call last):
  File "C:\Users\Acer Swift X\Downloads\iot project\YOLOv6\deploy\ONNX\export_onnx.py", line 15, in <module>
    from yolov6.models.yolo import *
ModuleNotFoundError: No module named 'yolov6'



In [2]:
"""
=============================================================================
SRTA 3353 - Machine Learning for IoT | Project 2
Student 4: Data Processing and Intelligence Engineer
=============================================================================
STEP 2: Feature Extraction
-----------------------------------------------------------------------------
Converts every .wav clip in combined_dataset/audio/ into a
(224, 224, 3) mel-spectrogram image saved as .npy  AND  writes a
ready-to-use features.npz that the training script loads directly.

Why mel-spectrogram?
  MobileNetV2 expects image-shaped input.  A log-scaled mel-spectrogram
  is a 2-D "image" of frequency content over time, proven effective for
  audio classification (e.g. VGGish, YAMNet, PANNs).

Preprocessing pipeline per clip:
  1. Load + resample to 22 050 Hz  (done in step 1)
  2. Short-Time Fourier Transform  (STFT)
  3. Map to mel scale  (128 mel bins)
  4. Convert power to dB  (log compression)
  5. Normalise to [0, 1]
  6. Resize to 224×224  (replicate across 3 channels for MobileNetV2)
=============================================================================
"""

import os
import numpy as np
import librosa
import cv2
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ─────────────────── CONFIG ───────────────────
METADATA_CSV    = "combined_dataset/metadata.csv"
AUDIO_DIR       = "combined_dataset/audio"
FEATURES_NPZ    = "combined_dataset/features.npz"

SAMPLE_RATE     = 22050
DURATION        = 4.0          # seconds
N_MFCC          = 40           # also extracted for lightweight models
N_MELS          = 128
HOP_LENGTH      = 512
N_FFT           = 2048
IMG_SIZE        = (224, 224)   # MobileNetV2 input
# ──────────────────────────────────────────────


def extract_melspec(y: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    """
    Returns a (224, 224, 3) float32 array:
      - log-mel spectrogram
      - normalised to [0, 1]
      - 3-channel (replicated) for MobileNetV2
    """
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Normalise to 0–1
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    mel_norm = mel_norm.astype(np.float32)

    # Resize to IMG_SIZE
    img = cv2.resize(mel_norm, IMG_SIZE, interpolation=cv2.INTER_LINEAR)

    # Stack to 3 channels
    img_3ch = np.stack([img, img, img], axis=-1)   # (224, 224, 3)
    return img_3ch


def extract_mfcc_vector(y: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    """
    Returns a flat 1-D feature vector of shape (N_MFCC * 3,):
      mean, std, and max of each MFCC coefficient over time.
    Useful for lightweight / rule-based threshold comparison.
    """
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    feat = np.concatenate([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        mfcc.max(axis=1),
    ])
    return feat.astype(np.float32)


# ─────────────────── MAIN ───────────────────
def main():
    df = pd.read_csv(METADATA_CSV)
    n  = len(df)

    X_img   = np.zeros((n, 224, 224, 3), dtype=np.float32)
    X_mfcc  = np.zeros((n, N_MFCC * 3),  dtype=np.float32)
    y_labels = np.zeros(n, dtype=np.int32)

    print(f"Extracting features for {n} clips ...")
    failed = 0

    for i, row in tqdm(df.iterrows(), total=n):
        path = os.path.join(AUDIO_DIR, row["filename"])
        try:
            y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
            target_len = int(SAMPLE_RATE * DURATION)
            if len(y) > target_len:
                y = y[:target_len]
            else:
                y = np.pad(y, (0, target_len - len(y)))

            X_img[i]  = extract_melspec(y)
            X_mfcc[i] = extract_mfcc_vector(y)
            y_labels[i] = int(row["label_id"])

        except Exception as e:
            print(f"  [WARN] {row['filename']}: {e}")
            failed += 1

    print(f"\nFeature extraction complete.  Failed: {failed}/{n}")
    print(f"X_img  shape : {X_img.shape}")   # (N, 224, 224, 3)
    print(f"X_mfcc shape : {X_mfcc.shape}")  # (N, 120)
    print(f"Labels shape : {y_labels.shape}")

    np.savez_compressed(
        FEATURES_NPZ,
        X_img=X_img,
        X_mfcc=X_mfcc,
        y=y_labels,
    )
    print(f"Saved → {FEATURES_NPZ}  ✓")


if __name__ == "__main__":
    main()

Extracting features for 3307 clips ...


100%|██████████| 3307/3307 [02:28<00:00, 22.22it/s]



Feature extraction complete.  Failed: 0/3307
X_img  shape : (3307, 224, 224, 3)
X_mfcc shape : (3307, 120)
Labels shape : (3307,)
Saved → combined_dataset/features.npz  ✓


In [3]:
"""
=============================================================================
SRTA 3353 - Machine Learning for IoT | Project 2
Student 4: Data Processing and Intelligence Engineer
=============================================================================
STEP 3: MobileNetV2 Audio Classifier — Training
-----------------------------------------------------------------------------
Model   : MobileNetV2 (pretrained ImageNet weights, transfer learning)
Input   : 224×224×3 mel-spectrogram images
Output  : 6-class softmax
            0 – glass_breaking
            1 – clock_alarm
            2 – gun_shot
            3 – siren
            4 – screaming
            5 – silence

Training strategy:
  Phase 1 (Freeze base, train head)  – 20 epochs
  Phase 2 (Fine-tune top 30 layers)  – 30 epochs + lower LR
=============================================================================
"""

import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import to_categorical

# ─────────────────── CONFIG ───────────────────
FEATURES_NPZ   = "combined_dataset/features.npz"
MODEL_DIR      = "models"
MODEL_PATH     = os.path.join(MODEL_DIR, "mobilenetv2_audio.h5")
TFLITE_PATH    = os.path.join(MODEL_DIR, "mobilenetv2_audio.tflite")
PLOT_DIR       = "plots"

NUM_CLASSES    = 6
IMG_SIZE       = (224, 224, 3)
BATCH_SIZE     = 32
EPOCHS_PHASE1  = 20
EPOCHS_PHASE2  = 30
RANDOM_SEED    = 42
TEST_SPLIT     = 0.15
VAL_SPLIT      = 0.15

CLASS_NAMES = [
    "glass_breaking", "clock_alarm", "gun_shot",
    "siren", "screaming", "silence"
]
# ──────────────────────────────────────────────

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR,  exist_ok=True)
tf.random.set_seed(RANDOM_SEED)


# ─────────────────── 1. Load features ───────────────────
print("[1] Loading features ...")
data = np.load(FEATURES_NPZ)
X    = data["X_img"].astype(np.float32)    # (N, 224, 224, 3)  already 0-1
y    = data["y"].astype(np.int32)          # (N,)

print(f"    X shape : {X.shape}")
print(f"    Classes : {np.bincount(y)}")

y_cat = to_categorical(y, NUM_CLASSES)

# Stratified split: 70% train / 15% val / 15% test
X_tmp,  X_test,  y_tmp,  y_test  = train_test_split(
    X, y_cat, test_size=TEST_SPLIT, stratify=y, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val  = train_test_split(
    X_tmp, y_tmp, test_size=VAL_SPLIT/(1-TEST_SPLIT),
    stratify=y_tmp.argmax(1), random_state=RANDOM_SEED)

print(f"    Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")


# ─────────────────── 2. Data augmentation ───────────────────
# Applied only during training; helps generalise to noisy real-world audio
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),          # mirrors spectrogram in time
    layers.RandomTranslation(0.05, 0.05),     # small time/freq shift
    layers.GaussianNoise(0.01),               # adds light noise
], name="augmentation")


# ─────────────────── 3. Build model ───────────────────
print("\n[2] Building MobileNetV2 model ...")

base_model = MobileNetV2(
    input_shape=IMG_SIZE,
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False   # Phase 1: frozen

inputs  = tf.keras.Input(shape=IMG_SIZE)
x       = augment(inputs, training=True)
x       = base_model(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(256, activation="relu")(x)
x       = layers.Dropout(0.4)(x)
x       = layers.Dense(128, activation="relu")(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs, name="MobileNetV2_AudioClassifier")
model.summary()


# ─────────────────── 4. Phase 1: Train head ───────────────────
print("\n[3] Phase 1 – training classification head (frozen base) ...")

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

cb_list = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True,
                            monitor="val_accuracy"),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=3,
                                monitor="val_loss", verbose=1),
    callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True,
                              monitor="val_accuracy", verbose=1),
]

hist1 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_PHASE1,
    batch_size=BATCH_SIZE,
    callbacks=cb_list,
    verbose=1,
)


# ─────────────────── 5. Phase 2: Fine-tune top layers ───────────────────
print("\n[4] Phase 2 – fine-tuning top 30 layers of MobileNetV2 ...")

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),   # lower LR for fine-tune
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

hist2 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_PHASE2,
    batch_size=BATCH_SIZE,
    callbacks=cb_list,
    verbose=1,
)


# ─────────────────── 6. Evaluate on test set ───────────────────
print("\n[5] Evaluating on test set ...")
model.load_weights(MODEL_PATH)
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"    Test accuracy : {acc*100:.2f}%")
print(f"    Test loss     : {loss:.4f}")

y_pred  = model.predict(X_test).argmax(axis=1)
y_true  = y_test.argmax(axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion matrix plot
cm  = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix – MobileNetV2 Audio Classifier")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "confusion_matrix.png"), dpi=150)
plt.close()
print(f"    Confusion matrix saved → {PLOT_DIR}/confusion_matrix.png")

# Training curves plot
all_acc     = hist1.history["accuracy"]     + hist2.history["accuracy"]
all_val_acc = hist1.history["val_accuracy"] + hist2.history["val_accuracy"]
all_loss    = hist1.history["loss"]         + hist2.history["loss"]
all_val_loss= hist1.history["val_loss"]     + hist2.history["val_loss"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(all_acc, label="Train"); ax1.plot(all_val_acc, label="Val")
ax1.axvline(EPOCHS_PHASE1, color="red", linestyle="--", label="Fine-tune start")
ax1.set_title("Accuracy"); ax1.legend(); ax1.set_xlabel("Epoch")

ax2.plot(all_loss, label="Train"); ax2.plot(all_val_loss, label="Val")
ax2.axvline(EPOCHS_PHASE1, color="red", linestyle="--", label="Fine-tune start")
ax2.set_title("Loss"); ax2.legend(); ax2.set_xlabel("Epoch")

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "training_curves.png"), dpi=150)
plt.close()
print(f"    Training curves saved → {PLOT_DIR}/training_curves.png")


# ─────────────────── 7. Export TFLite (for ESP32 / edge) ───────────────────
print("\n[6] Exporting TFLite model ...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # INT8 quantisation
tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f"    TFLite model saved → {TFLITE_PATH}  ({size_kb:.1f} KB)  ✓")
print("\nTraining complete ✓")

[1] Loading features ...
    X shape : (3307, 224, 224, 3)
    Classes : [  40   40  374  929 1724  200]
    Train: 2314 | Val: 496 | Test: 497

[2] Building MobileNetV2 model ...


Model: "MobileNetV2_AudioClassifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,624,710 (10.01 MB)

 Trainable params: 364,166 (1.39 MB)

...

73/73 ━━━━━━━━━━━━━━━━━━━━ 34s 460ms/step - accuracy: 0.9639 - loss: 0.1100 - val_accuracy: 0.9677 - val_loss: 0.1236 - learning_rate: 0.0010
Epoch 7/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 379ms/step - accuracy: 0.9776 - loss: 0.0830
Epoch 7: val_accuracy did not improve from 0.96774
73/73 ━━━━━━━━━━━━━━━━━━━━ 33s 450ms/step - accuracy: 0.9776 - loss: 0.0829 - val_accuracy: 0.9577 - val_loss: 0.1392 - learning_rate: 0.0010
Epoch 8/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step - accuracy: 0.9848 - loss: 0.0475
Epoch 8: val_accuracy did not improve from 0.96774
73/73 ━━━━━━━━━━━━━━━━━━━━ 32s 443ms/step - accuracy: 0.9848 - loss: 0.0476 - val_accuracy: 0.9577 - val_loss: 0.1309 - learning_rate: 0.0010
Epoch 9/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 371ms/step - accuracy: 0.9786 - loss: 0.0565
Epoch 9: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 9: val_accuracy did not improve from 0.96774
73/73 ━━━━━━━━━━━━━━━━━━━━ 32s 442ms/step - accuracy: 0.9786 - loss: 0.0566 - val_accu

73/73 ━━━━━━━━━━━━━━━━━━━━ 45s 621ms/step - accuracy: 0.9965 - loss: 0.0122 - val_accuracy: 0.9698 - val_loss: 0.1594 - learning_rate: 2.5000e-05
Epoch 14/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 514ms/step - accuracy: 0.9958 - loss: 0.0106
Epoch 14: val_accuracy improved from 0.96976 to 0.97379, saving model to models\mobilenetv2_audio.h5


73/73 ━━━━━━━━━━━━━━━━━━━━ 44s 604ms/step - accuracy: 0.9958 - loss: 0.0105 - val_accuracy: 0.9738 - val_loss: 0.1384 - learning_rate: 1.2500e-05
Epoch 15/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 523ms/step - accuracy: 0.9973 - loss: 0.0068
Epoch 15: val_accuracy did not improve from 0.97379
73/73 ━━━━━━━━━━━━━━━━━━━━ 45s 609ms/step - accuracy: 0.9973 - loss: 0.0068 - val_accuracy: 0.9718 - val_loss: 0.1261 - learning_rate: 1.2500e-05
Epoch 16/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 531ms/step - accuracy: 0.9981 - loss: 0.0069
Epoch 16: val_accuracy did not improve from 0.97379
73/73 ━━━━━━━━━━━━━━━━━━━━ 45s 616ms/step - accuracy: 0.9981 - loss: 0.0070 - val_accuracy: 0.9698 - val_loss: 0.1180 - learning_rate: 1.2500e-05
Epoch 17/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 499ms/step - accuracy: 0.9992 - loss: 0.0040
Epoch 17: val_accuracy did not improve from 0.97379
73/73 ━━━━━━━━━━━━━━━━━━━━ 42s 574ms/step - accuracy: 0.9992 - loss: 0.0040 - val_accuracy: 0.9718 - val_loss: 0.1114 - learning_rate: 1.2500e-05

INFO:tensorflow:Assets written to: C:\Users\ACERSW~1\AppData\Local\Temp\tmpsxqs1jt4\assets


Saved artifact at 'C:\Users\ACERSW~1\AppData\Local\Temp\tmpsxqs1jt4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2127284296336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2127284328928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2127284329456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2127284327520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2127284327696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2127885764912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2127885793408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2127885794464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2127885792000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2127885793232: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [21]:
"""
=============================================================================
SRTA 3353 - Machine Learning for IoT | Project 2
Student 4: Data Processing and Intelligence Engineer
=============================================================================
STEP 4: YOLOv6-nano Human Detection
-----------------------------------------------------------------------------
This script handles:
  A) Downloading YOLOv6-nano pretrained weights
  B) Running human (person) detection on images / video frames
  C) Preprocessing frames from ESP32-CAM (JPEG → RGB tensor)
  D) Postprocessing: filter "person" class, apply confidence threshold,
     compute presence score for decision logic

YOLOv6-nano is chosen because:
  - Fastest YOLOv6 variant (4.3M params, ~35 FPS on CPU)
  - mAP50 ≈ 35.0 on COCO – sufficient for indoor presence detection
  - Exports cleanly to ONNX for cross-platform inference

NOTE: YOLOv6 is from Meituan's repo.
      Install: pip install yolov6   OR   clone the official repo.
=============================================================================
"""

import os
import sys
import cv2
import numpy as np
import urllib.request
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional

# ─────────────────── CONFIG ───────────────────
WEIGHTS_DIR      = "models"
YOLO_WEIGHTS     = os.path.join(WEIGHTS_DIR, "yolov6n.pt")
YOLO_ONNX        = os.path.join(WEIGHTS_DIR, "yolov6n.onnx")

# YOLOv6-nano pretrained (COCO) – from official GitHub releases
WEIGHTS_URL = (
    "https://github.com/meituan/YOLOv6/releases/download/"
    "0.4.0/yolov6n.pt"
)

CONFIDENCE_THRESH = 0.45    # minimum detection confidence
NMS_IOU_THRESH    = 0.45    # NMS overlap threshold
PERSON_CLASS_ID   = 0       # COCO class 0 = person
IMG_SIZE          = 640     # YOLOv6 input size
PRESENCE_MIN_AREA = 0.02    # ignore boxes smaller than 2% of frame area
# ──────────────────────────────────────────────

os.makedirs(WEIGHTS_DIR, exist_ok=True)

COCO_CLASSES = [
    "person","bicycle","car","motorcycle","airplane","bus","train","truck",
    "boat","traffic light","fire hydrant","stop sign","parking meter","bench",
    "bird","cat","dog","horse","sheep","cow","elephant","bear","zebra",
    "giraffe","backpack","umbrella","handbag","tie","suitcase","frisbee",
    "skis","snowboard","sports ball","kite","baseball bat","baseball glove",
    "skateboard","surfboard","tennis racket","bottle","wine glass","cup",
    "fork","knife","spoon","bowl","banana","apple","sandwich","orange",
    "broccoli","carrot","hot dog","pizza","donut","cake","chair","couch",
    "potted plant","bed","dining table","toilet","tv","laptop","mouse",
    "remote","keyboard","cell phone","microwave","oven","toaster","sink",
    "refrigerator","book","clock","vase","scissors","teddy bear","hair drier",
    "toothbrush"
]


# ─────────────────── DATA CLASS ───────────────────
@dataclass
class Detection:
    bbox        : Tuple[int,int,int,int]   # x1, y1, x2, y2 (pixels)
    confidence  : float
    class_id    : int
    class_name  : str
    area_ratio  : float                    # bbox area / frame area  (0-1)


# ─────────────────────────────────────────────────────────────────────────────
# A.  Download weights
# ─────────────────────────────────────────────────────────────────────────────

def download_weights():
    if os.path.exists(YOLO_WEIGHTS):
        print(f"[Weights] Already exists: {YOLO_WEIGHTS}")
        return
    print(f"[Weights] Downloading YOLOv6-nano from:\n  {WEIGHTS_URL}")
    urllib.request.urlretrieve(WEIGHTS_URL, YOLO_WEIGHTS)
    print(f"[Weights] Saved → {YOLO_WEIGHTS}  ✓")


# ─────────────────────────────────────────────────────────────────────────────
# B.  Frame preprocessing (ESP32-CAM sends JPEG bytes over HTTP/UART)
# ─────────────────────────────────────────────────────────────────────────────

def preprocess_frame(frame_bgr: np.ndarray, target: int = IMG_SIZE
                     ) -> Tuple[np.ndarray, float, Tuple[int,int]]:
    """
    Letterbox-resize frame to (target, target) without distortion.
    Returns:
      blob      – (1, 3, target, target) float32 tensor  [0-1]
      scale     – resize scale factor  (used to map boxes back)
      padding   – (pad_w, pad_h) in pixels
    """
    h, w = frame_bgr.shape[:2]
    scale = min(target / h, target / w)
    new_h, new_w = int(h * scale), int(w * scale)

    resized = cv2.resize(frame_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas  = np.full((target, target, 3), 114, dtype=np.uint8)

    pad_h = (target - new_h) // 2
    pad_w = (target - new_w) // 2
    canvas[pad_h : pad_h + new_h, pad_w : pad_w + new_w] = resized

    # BGR → RGB, HWC → CHW, normalise
    rgb  = canvas[:, :, ::-1].astype(np.float32) / 255.0
    blob = np.expand_dims(rgb.transpose(2, 0, 1), axis=0)  # (1,3,640,640)

    return blob, scale, (pad_w, pad_h)


def postprocess_detections(
    raw_output : np.ndarray,     # (1, num_boxes, 85) – raw YOLOv6 output
    frame_h    : int,
    frame_w    : int,
    scale      : float,
    padding    : Tuple[int,int],
    conf_thresh: float = CONFIDENCE_THRESH,
    nms_iou    : float = NMS_IOU_THRESH,
) -> List[Detection]:
    """
    Parse raw ONNX output → list of Detection objects (person class only).
    """
    boxes_raw = raw_output[0]   # (num_boxes, 85)
    scores    = boxes_raw[:, 4] * boxes_raw[:, 5:]   # objectness * class_conf

    # Keep only person class
    person_scores = scores[:, PERSON_CLASS_ID]
    mask = person_scores >= conf_thresh

    if not mask.any():
        return []

    filtered      = boxes_raw[mask]
    person_conf   = person_scores[mask]

    # cx, cy, w, h → x1, y1, x2, y2  (in letterboxed coords)
    cx = filtered[:, 0]; cy = filtered[:, 1]
    bw = filtered[:, 2]; bh = filtered[:, 3]

    x1 = cx - bw / 2;  y1 = cy - bh / 2
    x2 = cx + bw / 2;  y2 = cy + bh / 2

    # Remove letterbox padding and rescale to original frame coords
    pad_w, pad_h = padding
    x1 = (x1 - pad_w) / scale;  y1 = (y1 - pad_h) / scale
    x2 = (x2 - pad_w) / scale;  y2 = (y2 - pad_h) / scale

    # Clip to frame
    x1 = np.clip(x1, 0, frame_w);  x2 = np.clip(x2, 0, frame_w)
    y1 = np.clip(y1, 0, frame_h);  y2 = np.clip(y2, 0, frame_h)

    # NMS (OpenCV)
    bboxes_cv = np.stack([x1, y1, x2-x1, y2-y1], axis=1).tolist()
    indices   = cv2.dnn.NMSBoxes(
        bboxes_cv, person_conf.tolist(), conf_thresh, nms_iou
    )

    detections = []
    frame_area = frame_h * frame_w

    for idx in (indices.flatten() if len(indices) > 0 else []):
        bx1, by1 = int(x1[idx]), int(y1[idx])
        bx2, by2 = int(x2[idx]), int(y2[idx])
        area_ratio = ((bx2 - bx1) * (by2 - by1)) / (frame_area + 1e-6)

        if area_ratio < PRESENCE_MIN_AREA:
            continue   # skip tiny ghost detections

        detections.append(Detection(
            bbox       = (bx1, by1, bx2, by2),
            confidence = float(person_conf[idx]),
            class_id   = PERSON_CLASS_ID,
            class_name = "person",
            area_ratio = area_ratio,
        ))

    return detections


# ─────────────────────────────────────────────────────────────────────────────
# C.  ONNX runtime inference wrapper
# ─────────────────────────────────────────────────────────────────────────────

class YOLOv6Detector:
    """
    Thin wrapper around onnxruntime for YOLOv6-nano inference.
    Use export_to_onnx() once to convert the .pt → .onnx, then
    instantiate this class with the .onnx path for fast inference.
    """

    def __init__(self, onnx_path: str = YOLO_ONNX):
        try:
            import onnxruntime as ort
        except ImportError:
            raise ImportError("pip install onnxruntime")

        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
        self.session = ort.InferenceSession(onnx_path, providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        print(f"[YOLOv6] Loaded ONNX model: {onnx_path}")

    def detect(self, frame_bgr: np.ndarray) -> List[Detection]:
        h, w = frame_bgr.shape[:2]
        blob, scale, padding = preprocess_frame(frame_bgr)
        raw = self.session.run(None, {self.input_name: blob})
        return postprocess_detections(raw[0], h, w, scale, padding)

    def draw(self, frame_bgr: np.ndarray,
             detections: List[Detection]) -> np.ndarray:
        """Draw bounding boxes on a copy of the frame."""
        out = frame_bgr.copy()
        for d in detections:
            x1, y1, x2, y2 = d.bbox
            cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f"person {d.confidence:.2f}"
            cv2.putText(out, label, (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
        return out


# ─────────────────────────────────────────────────────────────────────────────
# D.  Export .pt → .onnx  (run ONCE after pip install yolov6)
# ─────────────────────────────────────────────────────────────────────────────

def export_to_onnx():
    """
    Converts YOLOv6-nano .pt → .onnx.
    Requires the YOLOv6 repo cloned locally:
        git clone https://github.com/meituan/YOLOv6
        cd YOLOv6 && pip install -r requirements.txt
    """
    cmd = (
        f"python YOLOv6/export.py "
        f"--weights {YOLO_WEIGHTS} "
        f"--img {IMG_SIZE} "
        f"--device cpu "
        f"--simplify "
    )
    print(f"[Export] Running: {cmd}")
    os.system(cmd)


# ─────────────────────────────────────────────────────────────────────────────
# E.  Presence score helper  (used by decision logic in step5)
# ─────────────────────────────────────────────────────────────────────────────

def compute_presence_score(detections: List[Detection]) -> float:
    """
    Returns a single float in [0, 1] representing human presence confidence.
    Combines number of detections, their individual confidences,
    and bbox area relative to the frame.

    Score formula:
      presence = clip( mean_conf * sqrt(N) * (1 + mean_area), 0, 1 )

    - Rises quickly when 1 confident person detected
    - Rises further when multiple people detected
    - Penalises tiny far-away detections (small area_ratio)
    """
    if not detections:
        return 0.0

    n          = len(detections)
    mean_conf  = np.mean([d.confidence  for d in detections])
    mean_area  = np.mean([d.area_ratio  for d in detections])

    score = mean_conf * (n ** 0.5) * (1 + mean_area)
    return float(np.clip(score, 0.0, 1.0))


# ─────────────────────────────────────────────────────────────────────────────
# Quick test / demo
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # ── Demo: run on a single image passed as CLI arg ──
    if len(sys.argv) < 2:
        print("Usage: python step4_human_detection.py <image_path>")
        print("       e.g.  python step4_human_detection.py test.jpg")
        sys.exit(0)

    img_path = sys.argv[1]
    if not os.path.exists(YOLO_ONNX):
        print(f"[!] ONNX model not found at {YOLO_ONNX}")
        print("    Run export_to_onnx() after installing YOLOv6.")
        sys.exit(1)

    detector = YOLOv6Detector(YOLO_ONNX)
    frame    = cv2.imread(img_path)
    if frame is None:
        print(f"[!] Cannot read image: {img_path}")
        sys.exit(1)

    dets  = detector.detect(frame)
    score = compute_presence_score(dets)

    print(f"Detections   : {len(dets)}")
    for d in dets:
        print(f"  {d.class_name}  conf={d.confidence:.2f}  "
              f"bbox={d.bbox}  area={d.area_ratio:.3f}")
    print(f"Presence score: {score:.3f}")

    out_frame = detector.draw(frame, dets)
    out_path  = "detection_result.jpg"
    cv2.imwrite(out_path, out_frame)
    print(f"Result saved → {out_path}")

[!] ONNX model not found at models\yolov6n.onnx
    Run export_to_onnx() after installing YOLOv6.


SystemExit: 1